# Resume Training from Checkpoint

This notebook allows you to:
1. Load a checkpoint from `data/checkpoints/`
2. Validate the model to see current F1/accuracy
3. Resume training for additional epochs

In [13]:
import pandas as pd
import torch
import pickle
import numpy as np
import glob
import time
from datetime import datetime
from collections import defaultdict

from src.utils.paths import PROJECT_ROOT, get_data_folder
from src.utils.ontology_utils import load_ontology
from src.train.model import SimpleNN, WideNN
from src.train.loss import MarginalizationLoss
import torch.optim as optim

print("Imports complete.")

Imports complete.


## 1. Select Checkpoint

In [14]:
# List all available checkpoints
checkpoint_dir = PROJECT_ROOT / "data" / "checkpoints"
checkpoint_files = sorted(glob.glob(str(checkpoint_dir / "checkpoint_*.pt")))

print("Available checkpoints:")
print("="*80)
for i, ckpt_file in enumerate(checkpoint_files):
    ckpt_name = ckpt_file.split('/')[-1]
    print(f"  [{i}] {ckpt_name}")

if not checkpoint_files:
    print("\nNo checkpoints found in data/checkpoints/")
    print("Run training first using train.ipynb")
else:
    print("\n" + "="*80)
    print("Set CHECKPOINT_INDEX below to select which checkpoint to load")

Available checkpoints:
  [0] checkpoint_2026-01-16_15-34-24_epoch1.pt
  [1] checkpoint_2026-01-16_15-34-24_epoch10.pt
  [2] checkpoint_2026-01-16_15-34-24_epoch2.pt
  [3] checkpoint_2026-01-16_15-34-24_epoch3.pt
  [4] checkpoint_2026-01-16_15-34-24_epoch4.pt
  [5] checkpoint_2026-01-16_15-34-24_epoch5.pt
  [6] checkpoint_2026-01-16_15-34-24_epoch6.pt
  [7] checkpoint_2026-01-16_15-34-24_epoch7.pt
  [8] checkpoint_2026-01-16_15-34-24_epoch8.pt
  [9] checkpoint_2026-01-16_15-34-24_epoch9.pt
  [10] checkpoint_2026-01-20_15-06-12_epoch1.pt
  [11] checkpoint_2026-01-20_15-06-12_epoch2.pt
  [12] checkpoint_2026-01-20_15-06-12_epoch3.pt
  [13] checkpoint_2026-01-20_15-06-12_epoch4.pt
  [14] checkpoint_2026-01-20_15-06-12_epoch5.pt
  [15] checkpoint_2026-01-29_02-00-07_epoch1.pt
  [16] checkpoint_2026-01-29_02-00-07_epoch2.pt
  [17] checkpoint_2026-01-29_23-49-43_epoch1.pt
  [18] checkpoint_2026-02-02_00-35-32_epoch1.pt
  [19] checkpoint_2026-02-02_00-35-32_epoch2.pt
  [20] checkpoint_2026-02-

In [15]:
# ============================================================================
# CONFIGURATION - EDIT THESE VALUES
# ============================================================================

CHECKPOINT_INDEX = -1  # -1 = most recent, or use index from list above
ADDITIONAL_EPOCHS = 0  # How many more epochs to train (set to 0 for validation only)

# ============================================================================

checkpoint_path = checkpoint_files[CHECKPOINT_INDEX]
print(f"Selected checkpoint: {checkpoint_path.split('/')[-1]}")
print(f"Additional epochs to train: {ADDITIONAL_EPOCHS}")

Selected checkpoint: checkpoint_2026-02-02_00-35-32_epoch4.pt
Additional epochs to train: 0


## 2. Load Checkpoint

In [16]:
# Load checkpoint
print(f"Loading checkpoint from: {checkpoint_path}")
checkpoint = torch.load(checkpoint_path, map_location='cpu')

# Extract metadata
DATE = checkpoint['date_preprocessed']
ROOT_CL_ID = checkpoint.get('root_cl_id', 'CL:0000000')
input_dim = checkpoint['input_dim']
output_dim = checkpoint['output_dim']
completed_epochs = checkpoint['epoch']
saved_model_class = checkpoint.get('model_class', 'SimpleNN')
batch_loss_history = checkpoint.get('batch_loss_history', [])
epoch_times = checkpoint.get('epoch_times', [])

print("\n" + "="*80)
print("CHECKPOINT INFORMATION")
print("="*80)
print(f"Model architecture: {saved_model_class}({input_dim} -> {output_dim})")
print(f"Completed epochs: {completed_epochs}")
print(f"Total batches processed: {len(batch_loss_history):,}")
print(f"Final training loss: {batch_loss_history[-1]:.4f}" if batch_loss_history else "N/A")
print(f"Total training time: {sum(epoch_times)/3600:.2f} hours" if epoch_times else "N/A")
print(f"Preprocessed date: {DATE}")
print(f"Root CL ID: {ROOT_CL_ID}")
print("="*80)

Loading checkpoint from: /home/jingqiao/real_McCell/data/checkpoints/checkpoint_2026-02-02_00-35-32_epoch4.pt

CHECKPOINT INFORMATION
Model architecture: WideNN(20080 -> 236)
Completed epochs: 4
Total batches processed: 728,436
Final training loss: 0.0439
Total training time: 218.47 hours
Preprocessed date: 2026-01-29
Root CL ID: CL:0000000


## 3. Load Preprocessed Data

In [17]:
# Load data from the same date as the checkpoint
PROCESSED_DATA_DIR = get_data_folder(DATE, ROOT_CL_ID)
print(f"Loading data from: {PROCESSED_DATA_DIR}")

# Load the ontology
cl = load_ontology()

# Load DataFrames
marginalization_df = pd.read_csv(PROCESSED_DATA_DIR / f"{DATE}_marginalization_df.csv", index_col=0)
parent_child_df = pd.read_csv(PROCESSED_DATA_DIR / f"{DATE}_parent_child_df.csv", index_col=0)
exclusion_df = pd.read_csv(PROCESSED_DATA_DIR / f"{DATE}_exclusion_df.csv", index_col=0)

# Load mapping_dict
mapping_dict_df = pd.read_csv(PROCESSED_DATA_DIR / f"{DATE}_mapping_dict_df.csv", index_col=0)
mapping_dict = pd.Series(mapping_dict_df.iloc[:, 0].values, index=mapping_dict_df.index).to_dict()

# Load leaf and internal values
with open(PROCESSED_DATA_DIR / f"{DATE}_leaf_values.pkl", "rb") as fp:
    leaf_values = pickle.load(fp)
with open(PROCESSED_DATA_DIR / f"{DATE}_internal_values.pkl", "rb") as fp:
    internal_values = pickle.load(fp)

print(f"Loaded {len(mapping_dict)} cell types: {len(leaf_values)} leaf, {len(internal_values)} internal")

Loading data from: /home/jingqiao/real_McCell/data/processed/CL0000000_01-29
Loaded 369 cell types: 236 leaf, 133 internal


## 4. Initialize Model from Checkpoint

In [18]:
# Setup device
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
num_gpus = torch.cuda.device_count()
print(f"Using device: {device}")
print(f"Available GPUs: {num_gpus}")

# Create model
ModelClass = {"SimpleNN": SimpleNN, "WideNN": WideNN}[saved_model_class]
model = ModelClass(input_dim=input_dim, output_dim=output_dim)

# Load weights
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model weights from epoch {completed_epochs}")

# Wrap with DataParallel if multiple GPUs
if num_gpus > 1:
    print(f"Using DataParallel with {num_gpus} GPUs")
    model = torch.nn.DataParallel(model)

model = model.to(device)

# Initialize optimizer and load state
optimizer = optim.Adam(model.parameters(), lr=1e-4)
if 'optimizer_state_dict' in checkpoint:
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    print("Loaded optimizer state")

# Initialize loss function
loss_fn = MarginalizationLoss(
    marginalization_df=marginalization_df,
    parent_child_df=parent_child_df,
    exclusion_df=exclusion_df,
    leaf_values=leaf_values,
    internal_values=internal_values,
    mapping_dict=mapping_dict,
    leaf_weight=7.0,
    device=device
)

print("\nModel and optimizer ready.")

Using device: cuda:0
Available GPUs: 6
Loaded model weights from epoch 4
Using DataParallel with 6 GPUs
Loaded optimizer state

Model and optimizer ready.


## 5. Create DataLoaders

In [19]:
import tiledbsoma as soma
from tiledbsoma_ml import ExperimentDataset, experiment_dataloader

# Get all cell types
all_cell_values = list(mapping_dict.keys())

# Load gene list from BioMart
print("Loading protein-coding genes from BioMart...")
biomart_path = PROJECT_ROOT / "hpc_workaround/data/mart_export.txt"
biomart = pd.read_csv(biomart_path)
coding_only = biomart[biomart['Gene type'] == 'protein_coding']
gene_list = coding_only['Gene stable ID'].tolist()
print(f"Loaded {len(gene_list)} protein-coding genes")

# Create filter strings
var_value_filter = f"feature_id in {gene_list}"
obs_value_filter = f'assay == "10x 3\' v3" and is_primary_data == True and cell_type_ontology_term_id in {all_cell_values}'

# Open SOMA database
soma_uri = "/scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens"
print(f"\nOpening SOMA database at: {soma_uri}")
experiment = soma.open(soma_uri, mode="r")

# Create datasets
with experiment.axis_query(
    measurement_name="RNA",
    obs_query=soma.AxisQuery(value_filter=obs_value_filter),
    var_query=soma.AxisQuery(value_filter=var_value_filter),
) as query:
    experiment_dataset = ExperimentDataset(
        query,
        obs_column_names=["cell_type_ontology_term_id"],
        layer_name="raw",
        batch_size=256,
        shuffle=True,
        seed=111
    )

    train_dataset, val_dataset = experiment_dataset.random_split(0.8, 0.2, seed=42)
    
    print(f'\nTraining set: {len(train_dataset)} batches')
    print(f'Validation set: {len(val_dataset)} batches')

train_dataloader = experiment_dataloader(train_dataset, num_workers=2)
val_dataloader = experiment_dataloader(val_dataset, num_workers=2)
print("DataLoaders ready.")

Loading protein-coding genes from BioMart...
Loaded 23262 protein-coding genes

Opening SOMA database at: /scratch/sigbio_project_root/sigbio_project25/jingqiao/mccell-single/soma_db_homo_sapiens

Training set: 182109 batches
Validation set: 45528 batches
DataLoaders ready.


## 6. Validate Current Checkpoint

In [20]:
from sklearn.metrics import f1_score, precision_score, recall_score

def run_validation(model, val_dataloader, loss_fn, mapping_dict, leaf_values, internal_values, 
                   marginalization_df, device, val_batches=200):
    """
    Run validation and return metrics.
    """
    model.eval()
    
    # Setup
    leaf_indices_set = {mapping_dict[cid] for cid in leaf_values}
    internal_indices_set = {mapping_dict[cid] for cid in internal_values}
    
    # Marginalization tensor for internal node evaluation
    marginalization_tensor = torch.FloatTensor(
        marginalization_df.loc[internal_values, leaf_values].values
    ).to(device)
    internal_idx_to_position = {mapping_dict[cid]: i for i, cid in enumerate(internal_values)}
    
    # Collect predictions
    leaf_predictions = []
    leaf_labels = []
    internal_predictions = []
    val_losses = []
    
    total_samples = 0
    leaf_samples = 0
    internal_samples = 0
    
    with torch.no_grad():
        for i, (X_batch, obs_batch) in enumerate(val_dataloader):
            if i >= val_batches:
                break
            
            X_batch = torch.from_numpy(X_batch).float()
            X_batch = torch.log1p(X_batch).to(device)
            
            label_strings = obs_batch["cell_type_ontology_term_id"]
            y_batch = torch.tensor([mapping_dict[term] for term in label_strings],
                                  device=device, dtype=torch.long)
            
            total_samples += len(y_batch)
            
            # Process LEAF samples
            is_leaf = torch.tensor([y.item() in leaf_indices_set for y in y_batch], device=device)
            if is_leaf.sum() > 0:
                X_leaf = X_batch[is_leaf]
                y_leaf = y_batch[is_leaf]
                leaf_samples += len(y_leaf)
                
                outputs = model(X_leaf)
                total_loss, _, _ = loss_fn(outputs, y_leaf)
                val_losses.append(total_loss.item())
                
                preds = torch.argmax(outputs, dim=1)
                leaf_predictions.extend(preds.cpu().numpy())
                leaf_labels.extend(y_leaf.cpu().numpy())
            
            # Process INTERNAL samples
            is_internal = torch.tensor([y.item() in internal_indices_set for y in y_batch], device=device)
            if is_internal.sum() > 0:
                X_internal = X_batch[is_internal]
                y_internal = y_batch[is_internal]
                internal_samples += len(y_internal)
                
                outputs = model(X_internal)
                leaf_probs = torch.softmax(outputs, dim=1)
                internal_probs = torch.matmul(leaf_probs, marginalization_tensor.T)
                
                for j, true_idx in enumerate(y_internal):
                    pos = internal_idx_to_position[true_idx.item()]
                    pred_prob = internal_probs[j, pos].item()
                    internal_predictions.append(1 if pred_prob >= 0.5 else 0)
    
    # Calculate metrics
    leaf_predictions = np.array(leaf_predictions)
    leaf_labels = np.array(leaf_labels)
    internal_predictions = np.array(internal_predictions)
    
    leaf_accuracy = (leaf_predictions == leaf_labels).mean() if len(leaf_labels) > 0 else 0
    internal_accuracy = internal_predictions.mean() if len(internal_predictions) > 0 else 0
    
    micro_f1 = f1_score(leaf_labels, leaf_predictions, average='micro') if len(leaf_labels) > 0 else 0
    macro_f1 = f1_score(leaf_labels, leaf_predictions, average='macro') if len(leaf_labels) > 0 else 0
    weighted_f1 = f1_score(leaf_labels, leaf_predictions, average='weighted') if len(leaf_labels) > 0 else 0
    
    avg_loss = np.mean(val_losses) if val_losses else 0
    
    return {
        'total_samples': total_samples,
        'leaf_samples': leaf_samples,
        'internal_samples': internal_samples,
        'avg_loss': avg_loss,
        'leaf_accuracy': leaf_accuracy,
        'internal_accuracy': internal_accuracy,
        'micro_f1': micro_f1,
        'macro_f1': macro_f1,
        'weighted_f1': weighted_f1,
    }

print("Validation function defined.")

Validation function defined.


In [21]:
print("="*80)
print(f"VALIDATING CHECKPOINT (Epoch {completed_epochs})")
print("="*80)
print("Running validation on 200 batches...\n")

metrics = run_validation(
    model, val_dataloader, loss_fn, mapping_dict, 
    leaf_values, internal_values, marginalization_df, device,
    val_batches=200
)

print("\n" + "="*80)
print("VALIDATION RESULTS")
print("="*80)
print(f"Samples: {metrics['total_samples']:,} total ({metrics['leaf_samples']:,} leaf, {metrics['internal_samples']:,} internal)")
print(f"\nLoss:")
print(f"  Average Loss: {metrics['avg_loss']:.4f}")
print(f"\nLeaf Node Metrics:")
print(f"  Accuracy:    {metrics['leaf_accuracy']*100:.2f}%")
print(f"  Micro-F1:    {metrics['micro_f1']*100:.2f}%")
print(f"  Macro-F1:    {metrics['macro_f1']*100:.2f}%")
print(f"  Weighted-F1: {metrics['weighted_f1']*100:.2f}%")
print(f"\nInternal Node Metrics:")
print(f"  Accuracy:    {metrics['internal_accuracy']*100:.2f}%")
print("="*80)

VALIDATING CHECKPOINT (Epoch 4)
Running validation on 200 batches...


VALIDATION RESULTS
Samples: 51,200 total (30,893 leaf, 20,307 internal)

Loss:
  Average Loss: 0.4861

Leaf Node Metrics:
  Accuracy:    97.72%
  Micro-F1:    97.72%
  Macro-F1:    95.24%
  Weighted-F1: 97.72%

Internal Node Metrics:
  Accuracy:    80.62%


## 7. Resume Training (Optional)

Set `ADDITIONAL_EPOCHS` above to 0 to skip training.

In [ ]:
if ADDITIONAL_EPOCHS == 0:
    print("ADDITIONAL_EPOCHS = 0, skipping training.")
    print("Set ADDITIONAL_EPOCHS to a positive number to continue training.")
else:
    print(f"\n{'='*80}")
    print(f"RESUMING TRAINING")
    print(f"{'='*80}")
    print(f"Starting epoch: {completed_epochs + 1}")
    print(f"Ending epoch: {completed_epochs + ADDITIONAL_EPOCHS}")
    print(f"Training batches per epoch: {len(train_dataset):,}")
    print(f"{'='*80}\n")
    
    # Use existing checkpoint's run timestamp for continuity
    ckpt_name = checkpoint_path.split('/')[-1]
    # Extract timestamp from checkpoint name: checkpoint_YYYY-MM-DD_HH-MM-SS_epochN.pt
    run_timestamp = '_'.join(ckpt_name.replace('checkpoint_', '').split('_')[:2])
    
    batches_per_epoch = len(train_dataset)
    
    for epoch in range(completed_epochs, completed_epochs + ADDITIONAL_EPOCHS):
        model.train()
        epoch_start_time = time.time()
        print(f'\n{"="*80}')
        print(f'EPOCH {epoch + 1}/{completed_epochs + ADDITIONAL_EPOCHS}')
        print(f'{"="*80}')
        
        epoch_losses = []
        
        for i, (X_batch, obs_batch) in enumerate(train_dataloader):
            if i >= batches_per_epoch:
                break
            
            # Data preparation
            X_batch = torch.from_numpy(X_batch).float()
            X_batch = torch.log1p(X_batch).to(device)
            
            label_strings = obs_batch["cell_type_ontology_term_id"]
            y_batch = torch.tensor([mapping_dict[term] for term in label_strings], 
                                  device=device, dtype=torch.long)
            
            # Training step
            optimizer.zero_grad()
            outputs = model(X_batch)
            total_loss, loss_leafs, loss_parents = loss_fn(outputs, y_batch)
            total_loss.backward()
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            batch_loss_history.append(total_loss.item())
            epoch_losses.append(total_loss.item())
            
            # Progress updates
            if (i + 1) % 1000 == 0:
                elapsed = time.time() - epoch_start_time
                avg_loss_recent = sum(epoch_losses[-1000:]) / len(epoch_losses[-1000:])
                print(f'  [Batch {i+1:5d}/{batches_per_epoch}] '
                      f'Loss: {total_loss.item():.4f} (avg: {avg_loss_recent:.4f}) | '
                      f'Elapsed: {elapsed/3600:.2f}h')
        
        epoch_time = time.time() - epoch_start_time
        epoch_times.append(epoch_time)
        avg_epoch_loss = sum(epoch_losses) / len(epoch_losses)
        
        print(f'\n--- Epoch {epoch + 1} Summary ---')
        print(f'  Time: {epoch_time/3600:.2f} hours')
        print(f'  Average loss: {avg_epoch_loss:.4f}')
        
        # Save checkpoint
        checkpoint_save_path = checkpoint_dir / f"checkpoint_{run_timestamp}_epoch{epoch+1}.pt"
        
        if isinstance(model, torch.nn.DataParallel):
            state_dict_to_save = model.module.state_dict()
        else:
            state_dict_to_save = model.state_dict()
        
        ckpt = {
            'epoch': epoch + 1,
            'model_state_dict': state_dict_to_save,
            'optimizer_state_dict': optimizer.state_dict(),
            'batch_loss_history': batch_loss_history.copy(),
            'epoch_times': epoch_times.copy(),
            'avg_epoch_loss': avg_epoch_loss,
            'input_dim': input_dim,
            'output_dim': output_dim,
            'date_preprocessed': DATE,
            'root_cl_id': ROOT_CL_ID,
            'model_class': saved_model_class,
        }
        torch.save(ckpt, checkpoint_save_path)
        print(f'  Checkpoint saved: {checkpoint_save_path.name}')
    
    print('\n' + '='*80)
    print('TRAINING COMPLETE')
    print('='*80)
    print(f'Total epochs: {completed_epochs + ADDITIONAL_EPOCHS}')
    print(f'Total time: {sum(epoch_times)/3600:.2f} hours')


RESUMING TRAINING
Starting epoch: 3
Ending epoch: 4
Training batches per epoch: 182,109


EPOCH 3/4
  [Batch  1000/182109] Loss: 0.0847 (avg: 0.4348) | Elapsed: 0.31h


## 8. Post-Training Validation

In [ ]:
if ADDITIONAL_EPOCHS > 0:
    print("="*80)
    print(f"POST-TRAINING VALIDATION (Epoch {completed_epochs + ADDITIONAL_EPOCHS})")
    print("="*80)
    print("Running validation on 200 batches...\n")
    
    # Need to recreate dataloader since we consumed it during training
    val_dataloader = experiment_dataloader(val_dataset, num_workers=2)
    
    post_metrics = run_validation(
        model, val_dataloader, loss_fn, mapping_dict, 
        leaf_values, internal_values, marginalization_df, device,
        val_batches=200
    )
    
    print("\n" + "="*80)
    print("COMPARISON: Before vs After Training")
    print("="*80)
    print(f"{'Metric':<20} {'Before':<15} {'After':<15} {'Change':<15}")
    print("-"*65)
    
    for name, key in [('Avg Loss', 'avg_loss'), 
                      ('Leaf Accuracy', 'leaf_accuracy'),
                      ('Micro-F1', 'micro_f1'),
                      ('Macro-F1', 'macro_f1'),
                      ('Internal Acc', 'internal_accuracy')]:
        before = metrics[key]
        after = post_metrics[key]
        if 'loss' in key.lower():
            change = before - after  # Lower is better
            print(f"{name:<20} {before:<15.4f} {after:<15.4f} {change:+.4f}")
        else:
            change = (after - before) * 100
            print(f"{name:<20} {before*100:<14.2f}% {after*100:<14.2f}% {change:+.2f}%")
    
    print("="*80)

## 9. Save Final Model

In [ ]:
if ADDITIONAL_EPOCHS > 0:
    save_dir = PROJECT_ROOT / "data" / "saved_models"
    save_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    cl_id_clean = ROOT_CL_ID.replace(":", "")
    model_name = f"model_{cl_id_clean}_{timestamp}"
    model_path = save_dir / f"{model_name}.pt"
    
    if isinstance(model, torch.nn.DataParallel):
        state_dict_to_save = model.module.state_dict()
    else:
        state_dict_to_save = model.state_dict()
    
    final_checkpoint = {
        'model_state_dict': state_dict_to_save,
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': completed_epochs + ADDITIONAL_EPOCHS,
        'batch_loss_history': batch_loss_history,
        'epoch_times': epoch_times,
        'num_gpus': torch.cuda.device_count(),
        'input_dim': input_dim,
        'output_dim': output_dim,
        'date_preprocessed': DATE,
        'root_cl_id': ROOT_CL_ID,
        'model_class': saved_model_class,
        'total_training_cells': len(train_dataset) * 256,
        'total_batches_processed': len(batch_loss_history),
    }
    
    torch.save(final_checkpoint, model_path)
    print(f"\nFinal model saved to: {model_path}")
    print(f"  Total epochs: {completed_epochs + ADDITIONAL_EPOCHS}")
    print(f"  Total batches: {len(batch_loss_history):,}")